# Buscar imágenes por su contenido: el germen de la búsqueda multimodal

**Facsímil 11 · IA multimodal y percepción** — capítulos 3 y 4
(CLIP y aprendizaje contrastivo; modelos visión-lenguaje).

La búsqueda multimodal —buscar imágenes con texto, o imágenes parecidas entre sí— se reduce a una idea
sorprendentemente simple: **convertir todo en vectores y medir distancias**. En este cuaderno montas un
buscador de imágenes que no mira nombres de archivo ni etiquetas, sino el **contenido**: conviertes cada
imagen en un **vector descriptor** (un resumen numérico de sus colores) y buscas las más parecidas a una
consulta midiendo distancias entre vectores. Es, en pequeño y simplificado, lo que hace CLIP cuando busca
«un perro en la nieve». Aquí usamos el color para que se vea la mecánica; CLIP usa vectores **aprendidos**
mucho más ricos, pero la idea es **idéntica**.

### Qué vas a aprender
- Que buscar por contenido es **convertir cada imagen en un vector** y comparar distancias.
- Qué es un **descriptor** y cómo un histograma de color resume el «aspecto» de una imagen.
- A medir parecido con la **similitud del coseno**, por qué se **normaliza** y su relación con la distancia.
- A leer una **matriz de similitud** de toda la colección y a medir la búsqueda con **precisión@k**.
- La **limitación** de describir solo por color y cómo **enriquecer** el descriptor con textura.
- Cómo sería un **espacio compartido texto-imagen** (la idea de CLIP), con una maqueta de juguete.

### Cómo se usa
Las celdas vienen **sin ejecutar**: pulsa ▶ de arriba abajo y verás nacer cada número y cada gráfico.

### Cuánto cuesta
Unos 18 minutos. CPU, sin claves.


> **Inteligencia artificial para gente curiosa** · facsímil interactivo
> 
> Web del facsímil: https://www.iaparagentecuriosa.686f6c61.dev/ · Autor: @686f6c61 · Fecha: 2026-06-26 · Versión 1.0
> 
> Este cuaderno acompaña al facsímil: ejecútalo de arriba abajo, lee cada celda de texto
> antes de correr la de código y detente en las salidas. La gracia no es que «salga», sino
> entender *por qué* sale.

## 1. La idea: todo a vectores, comparar distancias

Un buscador de texto no compara letras: compara *embeddings* (facsímiles 1 y 4). Un buscador de imágenes
hace lo mismo con imágenes. La receta multimodal completa es: pon **imágenes y palabras en el mismo
espacio de vectores**, y entonces «buscar una imagen con la frase *gato en la nieve*» es solo encontrar el
vector de imagen más cercano al vector de esa frase. Aquí daremos el primer paso —imagen a vector— con un
descriptor sencillo, para que la mecánica quede a la vista. Empezamos con una pequeña colección.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(2)

# Coleccion de imagenes con un color dominante y textura (ruido).
def imagen(color, ruido=0.12, tam=48):
    base = np.ones((tam, tam, 3)) * np.array(color)
    return np.clip(base + np.random.normal(0, ruido, (tam, tam, 3)), 0, 1)

COLECCION = {
    "atardecer (naranja)": imagen([0.95, 0.55, 0.15]),
    "bosque (verde)":      imagen([0.20, 0.60, 0.25]),
    "mar (azul)":          imagen([0.15, 0.45, 0.85]),
    "nieve (blanco)":      imagen([0.92, 0.92, 0.95]),
    "lava (rojo)":         imagen([0.85, 0.15, 0.12]),
    "campo (verde claro)": imagen([0.55, 0.80, 0.40]),
}
print(f"La coleccion tiene {len(COLECCION)} imagenes. El buscador NO leera estos nombres: solo pixeles.")
fig, axes = plt.subplots(1, 6, figsize=(12, 2.2))
for ax, (nombre, im) in zip(axes, COLECCION.items()):
    ax.imshow(im); ax.set_title(nombre, fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()


## 2. Convertir cada imagen en un vector: el descriptor de color

El **descriptor**: para cada canal de color (rojo, verde, azul) hacemos un pequeño histograma de cuánto
hay de cada intensidad. Concatenados, dan un vector que resume el «aspecto» de la imagen. Dos imágenes con
colores parecidos tendrán descriptores cercanos. Lo normalizamos para poder compararlos por **ángulo** (la
similitud del coseno), que se fija en la *forma* del descriptor y no en su tamaño bruto.


In [ ]:
def descriptor(im, bins=8):
    vec = []
    for canal in range(3):
        h, _ = np.histogram(im[:,:,canal], bins=bins, range=(0,1), density=True)
        vec.append(h)
    v = np.concatenate(vec)
    return v / (np.linalg.norm(v) + 1e-9)      # normalizado, para comparar por angulo (coseno)

VECTORES = {nombre: descriptor(im) for nombre, im in COLECCION.items()}
dim = len(next(iter(VECTORES.values())))
print(f"Cada imagen es ahora un vector de {dim} numeros (3 canales x 8 bins).")
print("Norma de cada descriptor (deberia ser 1):",
      ", ".join(f"{np.linalg.norm(v):.2f}" for v in VECTORES.values()))
print("Buscar 'parecido' es medir el angulo (coseno) entre estos vectores.")


## 3. Ver el descriptor: dos imágenes, dos huellas

Un descriptor es una **huella numérica**. Si dibujamos el de dos imágenes muy distintas (el atardecer
naranja y el mar azul), veremos que las barras se concentran en sitios diferentes: el naranja carga el
canal rojo en intensidades altas; el azul, el canal azul. Esa diferencia de forma es lo que el coseno
medirá.


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 2.6), sharey=True)
for a, nombre in zip(ax, ["atardecer (naranja)", "mar (azul)"]):
    a.bar(range(dim), VECTORES[nombre], color="gray")
    for s, c in [(0, "red"), (8, "green"), (16, "blue")]:        # marcar el tramo de cada canal
        a.axvspan(s-0.5, s+7.5, color=c, alpha=0.08)
    a.set_title(nombre, fontsize=9); a.set_xlabel("R | G | B  (8 bins por canal)")
ax[0].set_ylabel("peso (normalizado)")
plt.tight_layout(); plt.show()
print("Mismo tipo de huella, distinta forma: el coseno entre ambas sera bajo.")


## 4. La consulta: «encuéntrame algo como esto»

Creamos una imagen de consulta —un tono anaranjado de atardecer que **no** está en la colección— y
buscamos las más parecidas por **similitud del coseno** entre descriptores. El buscador no sabe nada de
«atardeceres»: solo compara vectores. Aun así, debería traer lo naranja primero y mandar al fondo lo más
lejano en color.


In [ ]:
consulta = imagen([0.90, 0.50, 0.20])     # un atardecer nuevo, no esta en la coleccion
q = descriptor(consulta)
ranking = sorted(VECTORES.items(), key=lambda kv: -float(q @ kv[1]))   # coseno = producto escalar (ya normalizado)

plt.figure(figsize=(2,2)); plt.imshow(consulta); plt.title("consulta"); plt.axis("off"); plt.show()
print("Mas parecidas a la consulta (de mayor a menor similitud):\n")
for nombre, v in ranking:
    sim = float(q @ v)
    print(f"  {nombre:<22} {sim:.3f}  {'#'*int(round(sim*40))}")


**Lo que ha pasado.** El buscador no entiende de atardeceres ni leyó ninguna etiqueta: convirtió la
consulta en un vector y la comparó con los de la colección. Lo más naranja (el atardecer) sube arriba; lo
más lejano en color cae al fondo. Cambia «histograma de color» por «un vector aprendido que entiende
objetos y escenas» y tienes CLIP: **la misma mecánica**, descriptores muchísimo más inteligentes.


## 5. El mapa completo: matriz de similitud

En vez de una sola consulta, comparamos **todas con todas**. La matriz de similitud resume la colección de
un vistazo: la diagonal vale 1 (cada imagen consigo misma), y fuera de ella se ven los parecidos. Fíjate
en el par más brillante fuera de la diagonal: el atardecer y la lava. Son escenas bien distintas —un cielo
y un volcán—, pero comparten una paleta cálida rojiza, así que para un descriptor de color se parecen
mucho. Eso anticipa la limitación de la sección 8: el color confunde cosas distintas del mismo tono.


In [ ]:
nombres = list(VECTORES.keys())
M = np.array([VECTORES[a] for a in nombres])
S = M @ M.T                                          # 6x6 de similitudes coseno
fig, ax = plt.subplots(figsize=(5, 4.4))
im = ax.imshow(S, cmap="viridis", vmin=0, vmax=1)
ax.set_xticks(range(len(nombres))); ax.set_xticklabels(nombres, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(nombres))); ax.set_yticklabels(nombres, fontsize=8)
for i in range(len(nombres)):
    for j in range(len(nombres)):
        ax.text(j, i, f"{S[i,j]:.2f}", ha="center", va="center",
                color="white" if S[i,j] < 0.6 else "black", fontsize=7)
fig.colorbar(im, label="parecido (coseno)"); ax.set_title("Similitud entre todas las imagenes")
plt.tight_layout(); plt.show()
iu = np.triu_indices(len(nombres), k=1)                 # solo el triangulo superior (pares, sin diagonal)
b = int(np.argmax(S[iu]))
i, j = int(iu[0][b]), int(iu[1][b])
print(f"Par mas parecido (sin la diagonal): {nombres[i]} y {nombres[j]} -> {float(S[i,j]):.2f}")
print("Dos escenas distintas con paleta calida parecida: el color, por si solo, las confunde.")


## 6. Por qué se normaliza: proporción, no cantidad

Normalizar el descriptor (dividir por su norma) tiene un motivo. Sin normalizar, una imagen con un color
muy dominante produce un vector «más grande», y el producto escalar premiaría la **magnitud** en vez del
**parecido de forma**. Con vectores de norma 1, el producto escalar es justo el **coseno** del ángulo:
mide hacia dónde apunta el vector, no cuánto mide. Lo comprobamos comparando rankings con y sin normalizar.


In [ ]:
def descriptor_bruto(im, bins=8):
    vec = []
    for canal in range(3):
        h, _ = np.histogram(im[:,:,canal], bins=bins, range=(0,1), density=True)
        vec.append(h)
    return np.concatenate(vec)                        # SIN normalizar

bruto = {n: descriptor_bruto(im) for n, im in COLECCION.items()}
qb = descriptor_bruto(consulta)
rank_norm  = [n for n, _ in sorted(VECTORES.items(), key=lambda kv: -float(q @ kv[1]))]
rank_bruto = [n for n, _ in sorted(bruto.items(),     key=lambda kv: -float(qb @ kv[1]))]
print("Normas SIN normalizar (varian mucho):",
      ", ".join(f"{np.linalg.norm(v):.1f}" for v in bruto.values()))
print("\nRanking con coseno (normalizado):", " > ".join(n.split(' ')[0] for n in rank_norm))
print("Ranking con producto bruto:      ", " > ".join(n.split(' ')[0] for n in rank_bruto))
print("\nNormalizar evita que 'el que tiene el vector mas grande' gane por tamano y no por parecido.")


## 7. Medir la búsqueda: precisión@k

«Salen cosas naranjas» está bien, pero ¿cómo lo medimos? Si decidimos qué imágenes son **relevantes** para
la consulta (las cálidas: atardecer y lava), podemos calcular la **precisión@k**: de los primeros $k$
resultados, qué fracción es relevante. Es la métrica con la que se evalúan los buscadores de verdad.


In [ ]:
relevantes = {"atardecer (naranja)", "lava (rojo)"}     # lo que consideramos 'caliente' para esta consulta
orden = [n for n, _ in ranking]
print("Consulta: un tono calido. Relevantes:", ", ".join(sorted(relevantes)), "\n")
for k in [1, 2, 3]:
    top = orden[:k]
    aciertos = sum(1 for n in top if n in relevantes)
    print(f"  precision@{k} = {aciertos}/{k} = {aciertos/k:.2f}   (top-{k}: {', '.join(t.split(' ')[0] for t in top)})")
print("\nElegir k y el conjunto relevante es como traduces 'buscar bien' a un numero comparable.")


## 8. La limitación del color (y por qué CLIP es más listo)

Nuestro descriptor solo mira **color**. Dos cosas del mismo color pero muy distintas le parecen iguales.
Lo demostramos con un caso límite: dos imágenes **verdes** con la misma tonalidad pero **texturas**
opuestas —una lisa (un césped) y otra con franjas marcadas (un seto)—. Para el color son casi idénticas,
aunque para una persona sean cosas distintas. Ahí es donde el color se queda corto.


In [ ]:
liso = imagen([0.30, 0.65, 0.30], ruido=0.03)         # verde uniforme
seto = imagen([0.30, 0.65, 0.30], ruido=0.03).copy()
seto[:, ::4] *= 0.45                                   # franjas verticales oscuras: mucha textura
sim_color = float(descriptor(liso) @ descriptor(seto))
print(f"Similitud SOLO color entre el verde liso y el verde con franjas: {sim_color:.3f}")
print("Casi 1: el descriptor de color los da por iguales, aunque uno sea liso y el otro rayado.")
fig, ax = plt.subplots(1, 2, figsize=(5, 2.6))
ax[0].imshow(liso); ax[0].set_title("verde liso");    ax[0].axis("off")
ax[1].imshow(seto); ax[1].set_title("verde rayado");  ax[1].axis("off")
plt.tight_layout(); plt.show()


## 9. Enriquecer el descriptor: añadir textura

¿Y si al descriptor le sumamos una pista de **textura**? Una medida sencilla: cuánta variación hay entre
píxeles vecinos (los bordes). El césped liso tiene bordes suaves; el seto rayado, muchos bordes verticales.
Al concatenar color **y** textura, las dos imágenes verdes dejan de parecer iguales. Es exactamente la
intuición que lleva a CLIP: un descriptor **más rico** captura lo que el color solo no ve.


In [ ]:
def bordes(im):
    g = im.mean(axis=2)
    gx = np.abs(np.diff(g, axis=1)).mean()             # variacion horizontal (bordes verticales)
    gy = np.abs(np.diff(g, axis=0)).mean()             # variacion vertical
    return np.array([gx, gy])

def descriptor_rico(im, peso_textura=6.0):
    color = descriptor(im)
    text = bordes(im) * peso_textura
    v = np.concatenate([color, text])
    return v / (np.linalg.norm(v) + 1e-9)

print("Bordes (gx, gy) del verde liso: ", np.round(bordes(liso), 3))
print("Bordes (gx, gy) del verde rayado:", np.round(bordes(seto), 3), " <- mucho gx por las franjas")
sim_rico = float(descriptor_rico(liso) @ descriptor_rico(seto))
print(f"\nSimilitud SOLO color:           {sim_color:.3f}  (casi iguales)")
print(f"Similitud color + textura:      {sim_rico:.3f}  (ahora si se distinguen)")
print("Un descriptor mas rico ve diferencias que el color, por si solo, se pierde.")


## 10. La idea de CLIP: texto e imagen en el mismo espacio

Hasta aquí hemos comparado imágenes con imágenes. La magia de CLIP es poner **palabras e imágenes en el
mismo espacio**, para buscar fotos escribiendo. No tenemos CLIP aquí, pero podemos hacer una **maqueta de
juguete**: describimos cada imagen por tres ejes semánticos sencillos —calidez, claridad y saturación— y
escribimos consultas de texto como direcciones en esos mismos ejes. Buscar «algo cálido» pasa a ser hallar
la imagen cuyo vector apunta hacia ese lado. Es burdo comparado con CLIP, pero la **mecánica es la misma**.


In [ ]:
def concepto(im):
    r, g, b = im.reshape(-1, 3).mean(axis=0)
    calidez = r - b                                    # calido (rojo/naranja) vs frio (azul)
    claridad = (r + g + b) / 3                          # claro vs oscuro
    saturacion = float(im.reshape(-1, 3).max(axis=1).mean() - im.reshape(-1, 3).min(axis=1).mean())
    return np.array([calidez, claridad, saturacion])

nombres = list(COLECCION.keys())
C = np.array([concepto(im) for im in COLECCION.values()])
C = (C - C.mean(axis=0)) / (C.std(axis=0) + 1e-9)       # estandarizar los tres ejes
Cn = C / (np.linalg.norm(C, axis=1, keepdims=True) + 1e-9)

consultas_texto = {
    "algo calido":   np.array([1.0,  0.0, 0.0]),
    "algo luminoso": np.array([0.0,  1.0, 0.0]),
    "algo frio":     np.array([-1.0, 0.0, 0.0]),
}
for texto, vq in consultas_texto.items():
    vq = vq / np.linalg.norm(vq)
    sims = Cn @ vq
    top = nombres[int(np.argmax(sims))]
    print(f"Texto '{texto:<20}' -> imagen mas cercana: {top}")
print("\nEscribir texto y recuperar imagenes: eso es, en esencia, lo que hace CLIP (con vectores aprendidos).")


## 11. Robustez: ¿aguanta el ruido?

Una consulta real nunca es perfecta. ¿Y si la imagen de consulta llega con ruido (mala foto, compresión)?
Le metemos bastante ruido y comprobamos si el buscador sigue trayendo lo naranja arriba. Un buen
descriptor es **estable**: pequeños cambios en la entrada no deberían dar la vuelta al ranking.


In [ ]:
consulta_ruidosa = np.clip(consulta + np.random.normal(0, 0.25, consulta.shape), 0, 1)
qr = descriptor(consulta_ruidosa)
top_limpia = sorted(VECTORES.items(), key=lambda kv: -float(q  @ kv[1]))[0][0]
top_ruido  = sorted(VECTORES.items(), key=lambda kv: -float(qr @ kv[1]))[0][0]
print(f"Mejor resultado con la consulta limpia:   {top_limpia}")
print(f"Mejor resultado con la consulta ruidosa:  {top_ruido}")
print("Coincide:" , top_limpia == top_ruido, "-> el descriptor de color resiste bastante ruido.")
fig, ax = plt.subplots(1, 2, figsize=(4.4, 2.4))
ax[0].imshow(consulta);         ax[0].set_title("consulta");        ax[0].axis("off")
ax[1].imshow(consulta_ruidosa); ax[1].set_title("con ruido");       ax[1].axis("off")
plt.tight_layout(); plt.show()


## 12. Pruébalo tú

1. **Busca con otra consulta** (un azul de mar, un blanco de nieve). ¿Salen arriba las imágenes correctas?
   El mismo buscador sirve para cualquier consulta sin reentrenar nada.
2. **Sube los `bins`** del histograma (16, 32): descriptor más fino, ¿mejora el orden? Más detalle no
   siempre es mejor.
3. **Cambia el conjunto relevante** en la sección 7 (por ejemplo, solo `atardecer`) y recalcula la
   precisión@k. La métrica depende de qué decidas que «cuenta».
4. **Sube el `peso_textura`** del descriptor rico: ¿en qué momento la textura pesa tanto que dos colores
   muy distintos pero con la misma textura parecen iguales? Todo descriptor es un equilibrio.
5. **Añade consultas de texto** a la maqueta de CLIP (por ejemplo «algo verde y vivo»): tendrás que
   pensar hacia dónde apunta en los ejes calidez/claridad/saturación.


## 13. Errores comunes

- **Confundir buscar por nombre con buscar por contenido.** El primero mira metadatos; el segundo, los
  vectores de la propia imagen.
- **Usar un descriptor pobre para una tarea rica.** El color basta para «encuéntrame algo naranja», pero no
  para «encuéntrame un perro». Para eso, vectores aprendidos (CLIP).
- **Olvidar normalizar los vectores.** Sin normalizar, el producto escalar premia la magnitud y comparas
  «cantidad» en vez de «forma».
- **No medir.** Sin una métrica como precisión@k, «parece que funciona» no es una afirmación comprobable.
- **Creer que CLIP es magia.** Es la misma idea de este cuaderno —imagen a vector, comparar distancias—,
  solo que con un vector aprendido sobre millones de pares imagen-texto, y con texto e imagen en el mismo
  espacio.


## 14. Qué te llevas

- Buscar por contenido es **convertir cada imagen en un vector** y medir distancias: el corazón de la
  búsqueda multimodal y de los buscadores de imágenes modernos.
- La **similitud del coseno** (con vectores **normalizados**) mide parecido de forma; la **matriz de
  similitud** y la **precisión@k** te dejan ver y medir la búsqueda.
- Aquí el descriptor era el **color**; su límite (no entiende objetos ni texturas) es justo lo que se
  ataja con descriptores **más ricos** y, en última instancia, con los vectores **aprendidos** de CLIP.
- «Todo a vectores, comparar por distancia», con texto e imagen en el **mismo espacio**, es la idea que
  sostiene lo multimodal (facsímiles 1 y 4).

**Para seguir:** el capítulo 3 explica cómo CLIP aprende esos vectores alineando texto e imagen; el 6, cómo
se construye un RAG multimodal que recupera páginas, imágenes y tablas con evidencia.


---

### Ficha del cuaderno

- **Obra:** *Inteligencia artificial para gente curiosa* (facsímil interactivo).
- **Web:** https://www.iaparagentecuriosa.686f6c61.dev/
- **Autor:** @686f6c61
- **Fecha:** 2026-06-26
- **Versión:** 1.0

*Material pedagógico. Las salidas que ves son reales: se generan al ejecutar el código, no están escritas a mano. Si cambias algo, cambiarán: esa es la idea.*